# DSML Mini Project: Fake News Detection
### Course: Data Science and Machine Learning (DSML)

**Objective:** Build a machine learning classification model to detect whether a news article or headline is **REAL** or **FAKE** using Natural Language Processing (NLP) and supervised learning.

---

### Pipeline Steps:
1. **Load and Explore Dataset** (`news.csv`)
2. **Text Preprocessing** (Combining title and text, handling text formatting)
3. **Train-Test Split** (80% training data, 20% testing data)
4. **Feature Extraction** with **TF-IDF Vectorizer**
5. **Model Training** using **Passive Aggressive Classifier**
6. **Model Evaluation** (Accuracy, Confusion Matrix, Classification Report)
7. **Live Prediction Demonstration**

## 1. Import Required Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

## 2. Load and Inspect the Dataset

In [ ]:
df = pd.read_csv('news.csv')
print(f"Total Articles: {len(df)}")
df.head()

In [ ]:
# Check class distribution
print("Class Distribution:")
print(df['label'].value_counts())

## 3. Data Preparation
We concatenate the `title` and `text` columns to provide richer semantic context for classification.

In [ ]:
df['content'] = df['title'] + " " + df['text']
X = df['content']
y = df['label']
print("Sample content:")
print(X.iloc[0][:150] + "...")

## 4. Train-Test Split (80% Train, 20% Test)
We use `stratify=y` to ensure an equal proportion of REAL and FAKE news in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set size: {len(X_train)}")
print(f"Testing set size:  {len(X_test)}")

## 5. Feature Extraction using TF-IDF
**TF-IDF** stands for **Term Frequency - Inverse Document Frequency**.
- **TF:** Measures how frequently a word appears in a document.
- **IDF:** Diminishes the weight of words that occur very frequently across all documents (e.g., 'the', 'is').
- Removes common English stop words (`stop_words='english'`).

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.85)

# Fit on training data and transform both train and test
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"Number of unique features (words): {len(tfidf_vectorizer.get_feature_names_out())}")

## 6. Train the Passive Aggressive Classifier
The **Passive Aggressive (PA)** algorithm is an online learning algorithm.
- **Passive:** If the prediction is correct, keep the model weights unchanged.
- **Aggressive:** If the prediction is wrong, aggressively update weights to fix the error.

In [ ]:
model = PassiveAggressiveClassifier(max_iter=50, random_state=42)
model.fit(X_train_tfidf, y_train)
print("Model training complete!")

## 7. Model Evaluation
We evaluate the model on the unseen test dataset using Accuracy, Confusion Matrix, and Classification Report.

In [ ]:
y_pred = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {acc * 100:.2f}%")

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['REAL', 'FAKE'])
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=['Actual REAL', 'Actual FAKE'], columns=['Predicted REAL', 'Predicted FAKE']))

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

## 8. Save Model Artifacts
Save the trained model and vectorizer to disk so they can be loaded by the web app.

In [ ]:
joblib.dump(model, 'model.pkl')
joblib.dump(tfidf_vectorizer, 'vectorizer.pkl')
print("Artifacts saved as 'model.pkl' and 'vectorizer.pkl'!")

## 9. Live Testing Function
Test on any custom input string to verify real-time classification.

In [ ]:
def check_news(news_text):
    vec = tfidf_vectorizer.transform([news_text])
    pred = model.predict(vec)[0]
    return pred

test_1 = "Scientists at NASA discover evidence of ancient lakes on Mars."
test_2 = "Miracle boiling lemon water cures all viruses in 5 minutes! Share before it is banned!"

print(f"Text 1: '{test_1}'")
print(f"Result: {check_news(test_1)}\n")

print(f"Text 2: '{test_2}'")
print(f"Result: {check_news(test_2)}")